# Data Science & Database Assignment: Session 14
## Topic: Window Functions - Part 2 (`LAG`, `LEAD`, Cumulative Running Totals, and Rolling Window Frames)

**Student Name:** Atharv Chaudhari  
**Course:** Data Science & Database Fundamentals  
**Database Name:** `orders_window.db`  
**Core Concepts:** Value Window Functions (`LAG`, `LEAD`), Window Framing Specifications (`ROWS BETWEEN`), Cumulative Sums (`UNBOUNDED PRECEDING`), Rolling Moving Averages (`N PRECEDING AND CURRENT ROW`)  
**Status:** Completed & Pre-Executed  

---

### Window Framing Concepts
While ranking functions (`ROW_NUMBER`, `RANK`) assign an index to a row within its partition, **Value and Aggregate Window Functions** allow looking backward, forward, or across a dynamic sliding window of rows without needing a self-join:

```text
+-----------------------+------------------------------------------------------------------------------------+
| Clause / Function     | Behavioral Definition                                                              |
+-----------------------+------------------------------------------------------------------------------------+
| LAG(col, n)           | Fetches the value from 'n' rows prior within the partition. Default n = 1.         |
| LEAD(col, n)          | Fetches the value from 'n' rows ahead within the partition. Default n = 1.        |
| UNBOUNDED PRECEDING   | Window starts at the very first row of the current partition.                      |
| CURRENT ROW           | Window boundary terminates at the row currently being evaluated.                   |
| n PRECEDING           | Window includes exactly 'n' physical rows prior to the current row.                |
+-----------------------+------------------------------------------------------------------------------------+
```


## Task 1: Create 'Orders' Table & Populate Sample Food Orders
> **Problem Statement:** Create a table called `Orders` with columns: `order_id`, `user_id`, `order_date`, and `total_amount`. Insert at least 7 sample rows representing different users and dates, similar to how food orders appear in Zomato or Swiggy.

### Schema Architecture
- Entity capturing customer food orders placed across dates.
- Multiple transactions per user (`101`, `102`, `103`) to enable time-series partition analysis.


In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('orders_window.db')
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS Orders;")
cursor.execute("""
CREATE TABLE Orders (
    order_id INT PRIMARY KEY,
    user_id INT NOT NULL,
    order_date DATE NOT NULL,
    total_amount DECIMAL(10, 2) NOT NULL
);
""")

orders_data = [
    # User 101 Orders (Atharv)
    (1, 101, '2026-03-01', 350.00),
    (2, 101, '2026-03-05', 450.00),
    (3, 101, '2026-03-12', 600.00),
    (4, 101, '2026-03-20', 800.00),
    # User 102 Orders (Sneha)
    (5, 102, '2026-03-02', 200.00),
    (6, 102, '2026-03-08', 300.00),
    (7, 102, '2026-03-15', 250.00),
    (8, 102, '2026-03-22', 400.00),
    # User 103 Orders (Rahul)
    (9, 103, '2026-03-10', 500.00),
    (10, 103, '2026-03-18', 750.00)
]
cursor.executemany("INSERT INTO Orders VALUES (?, ?, ?, ?);", orders_data)
conn.commit()

df_raw = pd.read_sql_query("SELECT * FROM Orders;", conn)
display(df_raw)


,order_id,user_id,order_date,total_amount
0,1,101,2026-03-01,350
1,2,101,2026-03-05,450
2,3,101,2026-03-12,600
3,4,101,2026-03-20,800
4,5,102,2026-03-02,200
5,6,102,2026-03-08,300
6,7,102,2026-03-15,250
7,8,102,2026-03-22,400
8,9,103,2026-03-10,500
9,10,103,2026-03-18,750


## Task 2: Previous Order Amount Using LAG()
> **Problem Statement:** Write a SQL query using the `LAG()` function to show each user's `order_id`, `order_date`, and the `total_amount` of their previous order (if any), ordered by user and date.  
> *Hint: Use PARTITION BY user_id and ORDER BY order_date in your window function.*

### Conceptual Explanation
`LAG(total_amount, 1)` inspects the preceding row within the current user's partition.
- For each user's initial transaction, `LAG()` evaluates to `NULL` (rendered as `NaN`).
- We can compute the spending delta: `total_amount - LAG(total_amount, 1)` to track order-over-order spend growth.


In [2]:
task2_query = """
SELECT 
    user_id,
    order_id,
    order_date,
    total_amount,
    LAG(total_amount, 1) OVER (
        PARTITION BY user_id 
        ORDER BY order_date ASC
    ) AS previous_order_amount,
    ROUND(
        total_amount - LAG(total_amount, 1) OVER (
            PARTITION BY user_id 
            ORDER BY order_date ASC
        ), 2
    ) AS spend_difference
FROM Orders
ORDER BY user_id ASC, order_date ASC;
"""

df_t2 = pd.read_sql_query(task2_query, conn)
display(df_t2)


,user_id,order_id,order_date,total_amount,previous_order_amount,spend_difference
0,101,1,2026-03-01,350,NaN,NaN
1,101,2,2026-03-05,450,350.0,100.0
2,101,3,2026-03-12,600,450.0,150.0
3,101,4,2026-03-20,800,600.0,200.0
4,102,5,2026-03-02,200,NaN,NaN
5,102,6,2026-03-08,300,200.0,100.0
6,102,7,2026-03-15,250,300.0,-50.0
7,102,8,2026-03-22,400,250.0,150.0
8,103,9,2026-03-10,500,NaN,NaN
9,103,10,2026-03-18,750,500.0,250.0


## Task 3: Next Order Amount Using LEAD()
> **Problem Statement:** Using the same Orders table, write a SQL query with the `LEAD()` function to display each `order_id`, `order_date`, and the next order's `total_amount` for the same user.

### Conceptual Explanation
`LEAD(total_amount, 1)` looks ahead to the subsequent row within the user partition:
- For the user's latest transaction, `LEAD()` returns `NULL` because no future order exists.
- This function is essential in churn and retention analytics to measure the time elapsed and value difference before the next repeat purchase.


In [3]:
task3_query = """
SELECT 
    user_id,
    order_id,
    order_date,
    total_amount,
    LEAD(total_amount, 1) OVER (
        PARTITION BY user_id 
        ORDER BY order_date ASC
    ) AS next_order_amount
FROM Orders
ORDER BY user_id ASC, order_date ASC;
"""

df_t3 = pd.read_sql_query(task3_query, conn)
display(df_t3)


,user_id,order_id,order_date,total_amount,next_order_amount
0,101,1,2026-03-01,350,450.0
1,101,2,2026-03-05,450,600.0
2,101,3,2026-03-12,600,800.0
3,101,4,2026-03-20,800,NaN
4,102,5,2026-03-02,200,300.0
5,102,6,2026-03-08,300,250.0
6,102,7,2026-03-15,250,400.0
7,102,8,2026-03-22,400,NaN
8,103,9,2026-03-10,500,750.0
9,103,10,2026-03-18,750,NaN


## Task 4: Cumulative Running Total per User
> **Problem Statement:** Write a SQL query to calculate the running total of `total_amount` for each user, showing `order_id`, `order_date`, `total_amount`, and a column `running_total` that accumulates the sum as you move through each user's orders.  
> *Hint: Use SUM(total_amount) OVER (PARTITION BY user_id ORDER BY order_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW).*

### Conceptual Explanation
The window frame specification `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`:
- Starts summing from the very first order of that user (`UNBOUNDED PRECEDING`).
- Accumulates continuously up to the `CURRENT ROW`.
- Automatically resets back to the initial transaction when transitioning across user partitions.


In [4]:
task4_query = """
SELECT 
    user_id,
    order_id,
    order_date,
    total_amount,
    SUM(total_amount) OVER (
        PARTITION BY user_id 
        ORDER BY order_date ASC 
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_total
FROM Orders
ORDER BY user_id ASC, order_date ASC;
"""

df_t4 = pd.read_sql_query(task4_query, conn)
display(df_t4)


,user_id,order_id,order_date,total_amount,running_total
0,101,1,2026-03-01,350,350
1,101,2,2026-03-05,450,800
2,101,3,2026-03-12,600,1400
3,101,4,2026-03-20,800,2200
4,102,5,2026-03-02,200,200
5,102,6,2026-03-08,300,500
6,102,7,2026-03-15,250,750
7,102,8,2026-03-22,400,1150
8,103,9,2026-03-10,500,500
9,103,10,2026-03-18,750,1250


## Task 5: 3-Order Moving Average Using Window Frame
> **Problem Statement:** Write a SQL query to calculate a 3-order moving average of `total_amount` for each user, showing `order_id`, `order_date`, `total_amount`, and `moving_avg` columns.  
> *Constraint: Use SUM() OVER() with ROWS BETWEEN 2 PRECEDING AND CURRENT ROW to compute the moving average.*

---

### Engineering Calculation
To strictly fulfill the constraint of computing the moving average using `SUM() OVER()` with `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW`:
$$	ext{Moving Average} = rac{\sum_{i=0}^{2} 	ext{total\_amount}_{t-i}}{	ext{Count of available rows in frame}}$$

By dividing `SUM(...)` by `COUNT(...)` over the same sliding frame, the query handles boundary conditions correctly (e.g. Row 1 divides by 1, Row 2 divides by 2, and Row 3+ divides by 3).


In [5]:
task5_query = """
SELECT 
    user_id,
    order_id,
    order_date,
    total_amount,
    ROUND(
        SUM(total_amount) OVER (
            PARTITION BY user_id 
            ORDER BY order_date ASC 
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) * 1.0 / 
        COUNT(total_amount) OVER (
            PARTITION BY user_id 
            ORDER BY order_date ASC 
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 
        2
    ) AS moving_avg
FROM Orders
ORDER BY user_id ASC, order_date ASC;
"""

df_t5 = pd.read_sql_query(task5_query, conn)
display(df_t5)

conn.close()


,user_id,order_id,order_date,total_amount,moving_avg
0,101,1,2026-03-01,350,350.00
1,101,2,2026-03-05,450,400.00
2,101,3,2026-03-12,600,466.67
3,101,4,2026-03-20,800,616.67
4,102,5,2026-03-02,200,200.00
5,102,6,2026-03-08,300,250.00
6,102,7,2026-03-15,250,250.00
7,102,8,2026-03-22,400,316.67
8,103,9,2026-03-10,500,500.00
9,103,10,2026-03-18,750,625.00


## Summary & Window Frame Reference

| Window Frame Syntax | Frame Scope Description |
| :--- | :--- |
| `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` | Cumulative accumulator from partition start to current row (Standard Running Total). |
| `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` | Sliding 3-row historical moving frame (Past 2 orders + current order). |
| `ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING` | 3-row centered window (Previous row, current row, next row). |
| `ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING` | Reverse cumulative accumulator from current row to partition end. |
